In [1]:
%pip install langchain langchain-groq faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq

BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / ".env")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

respuesta = llm.invoke("Hola groq")
print(respuesta.content)

¡Hola! ¿En qué puedo ayudarte hoy?


In [4]:
from pathlib import Path

prompt_1 = (BASE_DIR / "prompt_1.md").read_text(encoding="utf-8")
prompt_2 = (BASE_DIR / "prompt_2.md").read_text(encoding="utf-8")
prompt_3 = (BASE_DIR / "prompt_3.md").read_text(encoding="utf-8")
prompt_4 = (BASE_DIR / "prompt_4.md").read_text(encoding="utf-8")


In [5]:
%pip install -q langchain-text-splitters langchain-community langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from transformers import AutoTokenizer
from langchain_community.vectorstores import FAISS

politicas_path = BASE_DIR / "politicas_ecommerce_integral.md"
texto_politicas = politicas_path.read_text(encoding="utf-8")

headers = [
    ("##", "Subseccion"),
    ("###", "Clausula"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers,
    strip_headers=False,
)

md_docs = markdown_splitter.split_text(texto_politicas)

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=500,
    chunk_overlap=50,
)

final_docs = text_splitter.split_documents(md_docs)

# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001",
# )

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

vector_store = FAISS.from_documents(final_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"Total chunks: {len(final_docs)}")
print(final_docs)
print(f"Total chunks in vector store: {vector_store.index.ntotal}")

C:\Users\Alan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Alan\AppData\Local\Temp\ipykernel_1488\1123115276.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11413.03it/s]


Total chunks: 12
[Document(metadata={'Subseccion': 'ENVIOS, GARANTÍAS, DEVOLUCIONES Y PROCESOS DE FACTURACIÓN'}, page_content='# POLÍTICA INTEGRAL DE TÉRMINOS Y CONDICIONES DE COMERCIO ELECTRÓNICO  \n## ENVIOS, GARANTÍAS, DEVOLUCIONES Y PROCESOS DE FACTURACIÓN  \n**Última actualización: Octubre de 2026**  \nEl presente documento constituye el marco legal integral y vinculante que regula todas las transacciones comerciales, adquisición de productos, despacho de mercancías, reclamos de garantía, reembolsos y modificaciones fiscales efectuadas a través de nuestro canal digital de ventas. Al acceder a esta plataforma, navegar en ella o consolidar una orden de compra, el cliente (en adelante, "El Usuario") declara haber leído, comprendido y aceptado expresamente la totalidad de las cláusulas detalladas a continuación.  \n---'), Document(metadata={'Subseccion': 'SECCIÓN 1: POLÍTICA INTEGRAL DE ENVÍOS, DESPACHOS Y LOGÍSTICA'}, page_content='## SECCIÓN 1: POLÍTICA INTEGRAL DE ENVÍOS, DESPACHOS

In [7]:
def recuperar_contexto(mensaje_usuario: str, k: int = 4) -> str:
    retriever = vector_store.as_retriever(search_kwargs={"k": k})

    docs = retriever.invoke(mensaje_usuario)

    return "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )

contexto = recuperar_contexto(
    "Hola! Me di cuenta de que dejé guardada mi dirección vieja para la orden. Todavía no la despacharon, ¿estoy a tiempo de cambiar la calle y altura para el envío?",
    k=4
)

print("Contexto recuperado:\n", contexto)

Contexto recuperado:
 ### 1.2 Intentos de Entrega y Costos de Reprogramación
Para garantizar una distribución eficiente, el servicio logístico estándar contratado contempla parámetros estrictos de entrega:
*   **Primer Intento de Entrega:** Se efectuará en la dirección exacta señalada por El Usuario al momento de la compra. Es responsabilidad exclusiva de El Usuario asegurar que se encuentre un mayor de edad (18+ años) con identificación válida para recibir y firmar la conformidad del paquete.
*   **Falla en la Entrega (Ausencia o Datos Incompletos):** En caso de que no se configure el éxito de la entrega debido a la ausencia de moradores, falta de respuesta al timbre o teléfono de contacto, o inconsistencias severas en la dirección provista, el transportista registrará la incidencia con evidencia fotográfica o geolocalización. El paquete regresará temporalmente al centro de distribución regional o sucursal base.
*   **Costo de Reprogramación:** El Usuario deberá abonar de forma obliga

In [9]:
#Casos de Prueba

mensajes_prueba = [
    "Buenas tardes, compré recién por error unas zapatillas y necesito cancelar el pedido antes de que lo preparen para envío. Muchas gracias.",
    "Hola! Me di cuenta de que dejé guardada mi dirección vieja para la orden. Todavía no la despacharon, ¿estoy a tiempo de cambiar la calle y altura para el envío?",
    "Buenas tardes, me acaba de llegar el pedido de juego de tazas y una de las tazas llegó rota por el viaje. Les adjunto la foto para que me indiquen el cambio o reintegro.",
    "Compré una campera y me llegó con una mancha y una costura rota. Para colmo, revisé el resumen y me la cobraron en dos cuotas cuando elegí pago único. Quiero un cambio inmediato y que me solucionen el tema de la tarjeta.",
    "El ventilador que compré hace una semana hace un ruido horrible y no gira. Fui a la sucursal a reclamar y el empleado me trató re mal. Quiero que me devuelvan el dinero en efectivo o los denuncio."
]


In [ ]:
#llamada a llm
import json
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

def procesar_mensaje(mensaje_usuario: str, temperature: float = 1.0) -> dict:
    
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=temperature
    )

    prompt1 = ChatPromptTemplate.from_messages([
        SystemMessage(content=prompt_1),
        ("system", "Contexto de políticas: {contexto}"),
        ("human", "{mensaje_usuario}")
    ])

    prompt2 = ChatPromptTemplate.from_messages([
        SystemMessage(content=prompt_2),
        ("human", "{input_json}")
    ])

    prompt3 = ChatPromptTemplate.from_messages([
        SystemMessage(content=prompt_3),
        ("human", "{input_json}")
    ])

    prompt4 = ChatPromptTemplate.from_messages([
        SystemMessage(content=prompt_4),
        ("human", "{input_json}")
    ])

    json_parser = JsonOutputParser()
    str_parser = StrOutputParser()

    step_1 = prompt1 | llm | str_parser
    step_2 = prompt2 | llm | str_parser
    step_3 = prompt3 | llm | str_parser
    step_4 = prompt4 | llm | json_parser

    chain_politica_aplicada = (
        {
            "contexto": lambda text: recuperar_contexto(text, k=4),
            "mensaje_usuario": RunnablePassthrough()
        }
        | step_1
        | step_2
        | step_3
        | step_4
    )

    return chain_politica_aplicada.invoke(mensaje_usuario)


mensaje_experimento = mensajes_prueba[1]
print(
    "Mensaje seleccionado para el"
    f' experimento:\n"{mensaje_experimento}"\n'
)

resultado = procesar_mensaje(mensaje_experimento, 1)
print(resultado)

Mensaje seleccionado para el experimento:
"Hola! Me di cuenta de que dejé guardada mi dirección vieja para la orden. Todavía no la despacharon, ¿estoy a tiempo de cambiar la calle y altura para el envío?"

{'metadatos_clasificacion': {'categoria': 'Envío/Logística', 'nivel_urgencia': 'Baja'}, 'analisis_interno': {'cadena_de_pensamiento': '1. El cliente solicita modificar la dirección (calle y altura) de una orden que aún no ha sido despachada. No se encontró en el contexto una política que regule la modificación de datos de envío antes del despacho, por lo que se necesita revisión por parte del operador. 2. Al no existir una política específica aplicable, la acción adecuada es crear un ticket para que el operador gestione la solicitud. 3. La urgencia es Baja porque la solicitud de cambio de dirección no implica un riesgo económico, de seguridad ni un plazo vencido; puede resolverse mediante gestión estándar sin intervención inmediata. 4. La respuesta al cliente comunica que se abrirá u